<a href="https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# Setup Cell (Run this first)
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

## 1. Unit of analysis + time window

Unit of Analysis: One row represents exactly one piece of content (content_hash_id) for one specific client (client_hash_id) aggregated over a specific 20-day performance window.
Time Window: We are iterating on a mid-panel month—specifically March 2026 (2026-03-01 to 2026-03-31). We are deliberately excluding the final month (June 2026) to prevent test-set contamination.

In [5]:
# Verify the time window and row counts for the March 2026 slice
query_window = """
    SELECT
        MIN(report_date) as slice_start,
        MAX(report_date) as slice_end,
        COUNT(*) as total_daily_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
con.sql(query_window).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,slice_start,slice_end,total_daily_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

Much like defining a strict schema or model architecture for a backend service, we must categorize our fields explicitly to prevent data leakage:

*   **Context:** `client_hash_id`, `content_hash_id`. (Identifiers to group the data, never passed to the model).
*   **Features (The 5-Feature Frame):**
    1. `imp_prev10`: Impressions in the 10 days prior. Knowable at the decision moment because historical logs are strictly retroactive.
    2. `clk_prev10`: Clicks in the 10 days prior. Knowable at the decision moment because Search Console reports them retroactively.
    3. `pos_prev10`: Average position in the 10 days prior. Knowable at the decision moment from daily warehouse syncs.
    4. `weekend_imp_share`: Proportion of traffic that occurs on weekends. Knowable at the decision moment from timestamp metadata.
    5. `ctr_prev10`: Click-through rate in the 10 days prior. Knowable at the decision moment because it is derived directly from historical clicks and impressions.
*   **Label:** `is_declining`. A binary target proxy (1 if impressions in the *next* 10 days drop by >50% compared to the previous 10 days, else 0).
*   **Excluded:** Any row where `imp_prev10 = 0`. **Why:** Content with zero baseline traffic cannot mathematically "decline" by 50%. Including dead pages skews the target logic and wastes compute.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Build the 5-feature frame and apply the exclusion rule
feature_frame_query = """
    WITH bounds AS (
        SELECT CAST('2026-03-31' AS DATE) as end_d
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            -- Target Window (The "Future" we want to predict)
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 10 DAY THEN f.gsc_impressions ELSE 0 END) as target_imp_last10,

            -- Feature Window (The "Past" - strictly knowable at decision moment)
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY AND f.report_date > b.end_d - INTERVAL 20 DAY THEN f.gsc_impressions ELSE 0 END) as imp_prev10,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY AND f.report_date > b.end_d - INTERVAL 20 DAY THEN f.gsc_clicks ELSE 0 END) as clk_prev10,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY AND f.report_date > b.end_d - INTERVAL 20 DAY THEN f.gsc_avg_position END) as pos_prev10,

            -- Weekend Share
            SUM(CASE WHEN EXTRACT(ISODOW FROM f.report_date) IN (6, 7) AND f.report_date <= b.end_d - INTERVAL 10 DAY THEN f.gsc_impressions ELSE 0 END) / NULLIF(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY THEN f.gsc_impressions ELSE 0 END), 0) AS weekend_imp_share,

            -- Click-Through Rate (CTR)
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY AND f.report_date > b.end_d - INTERVAL 20 DAY THEN f.gsc_clicks ELSE 0 END) / NULLIF(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY AND f.report_date > b.end_d - INTERVAL 20 DAY THEN f.gsc_impressions ELSE 0 END), 0) as ctr_prev10

        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet' f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 20 DAY
          AND f.report_date <= b.end_d
        GROUP BY 1, 2

        -- The Excluded Rule: Drop anything with zero historical traffic
        HAVING imp_prev10 > 0
    )
    SELECT *,
        -- Define the label proxy
        CASE WHEN target_imp_last10 < (imp_prev10 * 0.5) THEN 1 ELSE 0 END as is_declining
    FROM windowed
"""

df_features = con.sql(feature_frame_query).df()
print(f"Verified rows surviving the exclusion filter: {len(df_features):,}")
df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified rows surviving the exclusion filter: 150,693


,client_hash_id,content_hash_id,target_imp_last10,imp_prev10,clk_prev10,pos_prev10,weekend_imp_share,ctr_prev10,is_declining
0,client_62f4a7e64f5e0096,content_4b0b52a8d50fed52,55.0,39.0,0.0,3.270370,0.205128,0.000000,0
1,client_62f4a7e64f5e0096,content_321e70654be4f5e1,126.0,237.0,0.0,26.590594,0.168776,0.000000,0
2,client_62f4a7e64f5e0096,content_d6d4819215c6ad21,109.0,192.0,0.0,4.484309,0.203125,0.000000,0
3,client_62f4a7e64f5e0096,content_b975e2dc286ac584,395.0,294.0,0.0,9.193480,0.187075,0.000000,0
4,client_62f4a7e64f5e0096,content_0a135a504d85961c,220.0,548.0,1.0,5.302552,0.279197,0.001825,1


## 4. Data limits

**The Leakage Trap:** If we accidentally included the `target_imp_last10` column (or left the raw `report_date` from the future window) in our final training features, the model would score perfectly because it literally contains the answer. We must strictly drop `target_imp_last10` before fitting the classifier.

**What this data can never tell us:**
*   **External Algorithmic Shifts:** We only see Search Console metrics. We do not know if a sudden decline was caused by a broader Google Core Algorithm update or a seasonal trend out of the client's control.
*   **Technical Reality:** We don't know if a page died because of poor SEO, or if the client simply deleted the page from their server resulting in an HTTP 404 error.
*   **Unbalanced Client History:** As seen in `dim_clients.gsc_data_start`, different clients were onboarded at different times. Older clients inherently have more robust historical features than newly onboarded ones.

In [7]:
# The Trap: Demonstrating how we strictly drop the future target to prevent leakage
# If we kept 'target_imp_last10', our model would achieve 100% fake accuracy.

honest_features = df_features.drop(columns=['target_imp_last10'])

print("Original columns (contains leakage):", list(df_features.columns))
print("Honest columns (leakage removed):", list(honest_features.columns))

Original columns (contains leakage): ['client_hash_id', 'content_hash_id', 'target_imp_last10', 'imp_prev10', 'clk_prev10', 'pos_prev10', 'weekend_imp_share', 'ctr_prev10', 'is_declining']
Honest columns (leakage removed): ['client_hash_id', 'content_hash_id', 'imp_prev10', 'clk_prev10', 'pos_prev10', 'weekend_imp_share', 'ctr_prev10', 'is_declining']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.